# 05. Overdue Prediction (Binary Classification)
This notebook trains and evaluates binary classification models (Logistic Regression vs Random Forest Classifier) to predict task late-completion risk.
It measures Accuracy, Precision, Recall, F1 Score, ROC-AUC, and extracts interpretable feature importance.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

sys.path.append(os.path.abspath('../backend'))
from app.analytics.queries import get_tasks_analytics_dataframe
from app.ml.features import extract_ml_dataset, FEATURE_COLUMNS
from app.ml.classification import train_and_evaluate_classifiers
from app.ml.evaluation import extract_feature_importance
from app.db.session import engine
from sqlalchemy.orm import sessionmaker

SessionLocal = sessionmaker(bind=engine)
db = SessionLocal()
raw_df = get_tasks_analytics_dataframe(db)
df, _ = extract_ml_dataset(raw_df)
print(f'Training dataset ready with {len(df)} records.')

## 1. Train and Evaluate Classification Models

In [ ]:
X = df[FEATURE_COLUMNS]
y = df['is_late']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

best_pipeline, metrics, comparison = train_and_evaluate_classifiers(X_train, y_train, X_test, y_test)
print(f'Selected Model: {comparison["selected_model"]}')
print('Metrics Comparison:')
print(pd.DataFrame(comparison['candidate_metrics']))

## 2. Feature Importance Analysis

In [ ]:
importance = extract_feature_importance(best_pipeline)
print('Classification Feature Importance:')
print(pd.DataFrame(importance))